# Exploratory Per-Ticker File-Order Inversions — DEBS 2022 Trading Data

**Source:** `data/src/debs2022-gc-trading-day-12-11-21.csv` (~4.8 GB, 56 M rows, trading day `12-11-2021`).

This notebook focuses on a single question: **is each ticker's event stream in time order, and how late are the events that are not?**

For every security `ID`, events are processed in **file (arrival) order** and compared against that ticker's own state:

- **out-of-order row** (called `late` in tables and plots): its system `Time` is earlier than the maximum `Time` already seen **for that same ticker**;
- **adjacent inversion** — its `Time` is earlier than the immediately preceding event of the same ticker;
- **inversion depth** (called `lateness`): `(prior per-ticker maximum) − (system Time)`, in seconds.

Only **price events** are analysed. A row is a price event if it carries a non-blank value in any price column: `Ask`, `Bid`, `Last`, `Mid price`, `Day's high`, `Day's low`, `Open`, `Close`, `Auction price`, `Current price`. Heartbeat (timestamp-only), volume-only, trading-time-only and static reference rows are ignored, as are sentinel `00:00:00.000` times.

**Not the replay or Flink late-event rate.** This analysis uses only the system `Time` (time of day, no date or zone). Its `Last` subset includes any nonblank string, even `0`, and it excludes midnight. In contrast, `DataLoader.extract` chooses `Trading date` over `Date` and `Trading time` over `Time`, independently when nonblank, then computes epoch milliseconds in `EVENT_TIME_ZONE` (Compose default `Europe/Berlin`). For example, with `Time=09:30:02.000` and `Trading time=09:29:58.000`, the two analyses order the row on different timestamps; valid midnight is not categorically excluded by the emitter. The default replay can emit quotes and housekeeping too, but `EventJson.parse` uses a finite, strictly positive `last` for replay trades. Flink assigns watermarks after filtering, using bounded out-of-orderness and idleness, and `PriceIndicators` judges EMA lateness against window boundaries and the configured policy. No per-ticker running maximum here models that behavior. See section 7 for actual replay accounting.

The summary streams PyArrow record batches; the timeline section makes a second pass and retains rows for the selected tickers.


## 0. Setup

Use a Jupyter Python kernel with `numpy`, `pandas`, `pyarrow`, and `matplotlib`. The cells look for the CSV at the path given above, relative to the current directory or an ancestor. Set `DEBS_MAX_ROWS` to a positive integer before starting the kernel to explore just a prefix.

In [ ]:
import os, time, gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pc
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["figure.dpi"] = 110

try:
    display
except NameError:          # running outside IPython (e.g. headless validation)
    display = print

CSV_PATH = "data/src/debs2022-gc-trading-day-12-11-21.csv"
if not os.path.exists(CSV_PATH):          # allow running from experiments/ or deeper
    for base in ("..", "../..", "../../.."):
        cand = os.path.join(base, CSV_PATH)
        if os.path.exists(cand):
            CSV_PATH = cand
            break

MAX_ROWS = int(os.environ["DEBS_MAX_ROWS"]) if os.environ.get("DEBS_MAX_ROWS") else None
BLOCK_SIZE = 1 << 25     # ~32 MiB CSV per record batch
TOP_K = 6                # tickers plotted in detail

print("pandas", pd.__version__, "| numpy", np.__version__, "| pyarrow", pa.__version__)
print("file size: %.2f GB" % (os.path.getsize(CSV_PATH) / 1e9), "| MAX_ROWS:", MAX_ROWS)


## 1. Reading the file

The CSV has a `#` comment preamble, a header line, a `#`-prefixed column-description line, and data rows with one trailing empty field (40 fields vs 39 header names). We skip the first 13 lines and supply explicit column names.

In [ ]:
COLS = [
    "ID", "SecType", "Date", "Time", "Ask", "Ask volume", "Bid", "Bid volume",
    "Ask time", "Day's high ask", "Close", "Currency", "Day's high ask time",
    "Day's high", "ISIN", "Auction price", "Day's low ask", "Day's low",
    "Day's low ask time", "Open", "Nominal value", "Last", "Last volume",
    "Trading time", "Total volume", "Mid price", "Trading date", "Profit",
    "Current price", "Related indices", "Day high bid time", "Day low bid time",
    "Open Time", "Last trade time", "Close Time", "Day high Time", "Day low Time",
    "Bid time", "Auction Time", "_extra",
]
# A row is a "price event" if any of these price columns is non-blank.
PRICE_COLS = ["Ask", "Bid", "Last", "Mid price", "Day's high", "Day's low",
              "Open", "Close", "Auction price", "Current price"]

CONVERT = pc.ConvertOptions(column_types={c: pa.string() for c in COLS})


def read_batches(path, block_size=BLOCK_SIZE, max_rows=MAX_ROWS):
    """Yield PyArrow record batches of the trading CSV, in file order."""
    ro = pc.ReadOptions(skip_rows=13, column_names=COLS,
                        block_size=block_size, use_threads=True)
    reader = pc.open_csv(path, read_options=ro, convert_options=CONVERT)
    total = 0
    for b in reader:
        if max_rows is not None and total >= max_rows:
            break
        if max_rows is not None and total + b.num_rows > max_rows:
            b = b.slice(0, max_rows - total)
        total += b.num_rows
        yield b


peek = pd.read_csv(CSV_PATH, skiprows=13, names=COLS, header=None,
                   nrows=5, dtype=str, keep_default_na=False)
peek[["ID", "SecType", "Date", "Time", "Ask", "Bid", "Last"]]


## 2. Per-ticker streaming pass

The pass is restricted to **price events** (`PRICE_COLS`); all other rows are skipped. For each batch we parse `Time` to seconds and, per ticker, compute the **running maximum** (watermark) and compare each event to the maximum of the events *before* it:

- `late  = time < running_max_before`
- `adjacent = time < previous_time`

Cross-batch state (`last`, `max`) is carried per ticker so results are independent of the batch size. Per-ticker totals are accumulated in a small dictionary (only ~5 k tickers).

In [ ]:
t0 = time.time()
n_rows = n_valid = n_reference = n_sentinel = n_nonprice = n_trade = n_quote = 0

# ID -> [n_events, n_late, n_adj, sum_late, max_late, first_time, last_time, watermark]
T = {}
state = {}          # ID -> (last_time, max_time)
late_sample = []    # down-sampled lateness values for the histogram

for b in read_batches(CSV_PATH):
    df = b.to_pandas()
    n_rows += len(df)

    secs = pd.to_timedelta(df["Time"], errors="coerce").dt.total_seconds().to_numpy()
    timed = ~np.isnan(secs)
    sentinel = timed & (secs == 0)          # 00:00:00.000 placeholder -> excluded
    price = df[PRICE_COLS].ne("").any(axis=1).to_numpy()   # keep only price events
    valid = timed & ~sentinel & price
    n_reference += int((~timed).sum())
    n_sentinel += int(sentinel.sum())
    n_nonprice += int((timed & ~sentinel & ~price).sum())
    n_trade += int((valid & df["Last"].ne("").to_numpy()).sum())
    n_quote += int((valid & (df["Ask"].ne("").to_numpy() | df["Bid"].ne("").to_numpy())).sum())
    if not valid.any():
        continue
    n_valid += int(valid.sum())

    sub = pd.DataFrame({"ID": df["ID"].to_numpy()[valid], "s": secs[valid]})
    g = sub.groupby("ID", sort=False)["s"]

    # Ticker watermarks carried from previous batches, broadcast to every row.
    wm_map = {k: v[1] for k, v in state.items()}
    last_map = {k: v[0] for k, v in state.items()}

    # running max before the current event = max(carried watermark, within-batch running max)
    cm = g.cummax()
    prev = cm.groupby(sub["ID"], sort=False).shift(1)
    prev = prev.fillna(-np.inf).to_numpy()
    prev = np.maximum(prev, sub["ID"].map(wm_map).fillna(-np.inf).to_numpy())

    late = sub["s"].to_numpy() < prev
    late_s = np.where(late, prev - sub["s"].to_numpy(), 0.0)

    # adjacent inversion vs. previous event of the same ticker
    adj = np.array((g.diff() < 0).to_numpy(), dtype=bool)
    firsts = g.head(1).index
    adj[firsts.to_numpy()] = (
        sub.loc[firsts, "s"].to_numpy() < sub.loc[firsts, "ID"].map(last_map).to_numpy())

    sub["late"], sub["adj"], sub["late_s"] = late, adj, late_s
    gb = sub.groupby("ID", sort=False)
    agg = pd.DataFrame({
        "n": gb.size(), "n_late": gb["late"].sum(), "n_adj": gb["adj"].sum(),
        "sum_late": gb["late_s"].sum(), "max_late": gb["late_s"].max(),
        "first": gb["s"].first(), "last": gb["s"].last(), "wm": gb["s"].max(),
    })

    for id_, r in agg.iterrows():
        rec = T.get(id_)
        if rec is None:
            T[id_] = [r.n, r.n_late, r.n_adj, r.sum_late, r.max_late, r["first"], r["last"], r.wm]
            state[id_] = (r["last"], r.wm)
        else:
            rec[0] += r.n; rec[1] += r.n_late; rec[2] += r.n_adj; rec[3] += r.sum_late
            rec[4] = max(rec[4], r.max_late); rec[5] = min(rec[5], r["first"]); rec[6] = r["last"]
            newmax = max(state[id_][1], r.wm)
            rec[7] = newmax
            state[id_] = (r["last"], newmax)

    if late.any():
        ls = late_s[late]
        late_sample.append(ls[::max(1, ls.size // 2000)])

    if time.time() - t0 > 10:
        print(f"  ... {n_rows:,} rows ({time.time()-t0:.0f}s)", flush=True)

print(f"scanned {n_rows:,} rows in {time.time()-t0:.1f}s")


In [ ]:
tickers = pd.DataFrame.from_dict(
    T, orient="index",
    columns=["n_events", "n_late", "n_adj", "sum_late", "max_late",
             "first_time", "last_time", "watermark"])
tickers.index.name = "ID"
tickers["late_rate"] = tickers.n_late / tickers.n_events
tickers["adj_rate"] = tickers.n_adj / tickers.n_events
tickers["mean_late"] = tickers.sum_late / tickers.n_late.replace(0, np.nan)
tickers = tickers.sort_values("n_late", ascending=False)

summary = pd.Series({
    "rows scanned": n_rows,
    "price events analysed": n_valid,
    "  ... rows with nonblank Last (not validated trades)": n_trade,
    "  ... quote (Ask/Bid)": n_quote,
    "non-price timed events ignored": n_nonprice,
    "reference rows (no Time) ignored": n_reference,
    "sentinel 00:00:00.000 ignored": n_sentinel,
    "tickers": len(tickers),
    "tickers with >=1 late event": int((tickers.n_late > 0).sum()),
    "tickers with >=1 adjacent inversion": int((tickers.n_adj > 0).sum()),
    "tickers fully time-ordered": int(((tickers.n_late == 0) & (tickers.n_adj == 0)).sum()),
    "total late events": int(tickers.n_late.sum()),
    "total adjacent inversions": int(tickers.n_adj.sum()),
    "overall late rate": tickers.n_late.sum() / tickers.n_events.sum(),
    "worst single lateness (s)": tickers.max_late.max(),
}, name="value")
summary.to_frame()


## 3. How widespread is per-ticker disorder?

The distribution below shows each ticker's file-order inversion rate relative to its prior maximum.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].hist(tickers.late_rate, bins=50, color="#c44e52")
ax[0].set_yscale("log")
ax[0].set_xlabel("late-event rate per ticker"); ax[0].set_ylabel("tickers (log)")
ax[0].set_title("Per-ticker late-event rate")

sc = ax[1].scatter(tickers.n_events, tickers.n_late, s=8, alpha=0.4, color="#4c72b0")
ax[1].set_xscale("log"); ax[1].set_yscale("symlog")
ax[1].set_xlabel("events per ticker (log)"); ax[1].set_ylabel("late events")
ax[1].set_title("Late events vs. ticker size")
plt.tight_layout(); plt.show()


**Interpretation.** Per-ticker lateness is bimodal. Many "late" price events are simply a ticker's normal stream sitting *below* a watermark that was raised by an earlier **future-dated** price event, so a single outlier can mark thousands of later events as late — some tickers show ~100% of their price events flagged while having only a handful of adjacent inversions. The **adjacent-inversion** count is the outlier-robust measure of genuine local reordering; both are reported so the two regimes stay visible.

## 4. Worst offenders

Tickers ranked by number of late events, and by the single largest lateness.

In [ ]:
cols = ["n_events", "n_late", "late_rate", "n_adj", "max_late", "mean_late"]
print("Top tickers by number of late events")
display(tickers.head(15)[cols].round(3))

print("Top tickers by single worst lateness (seconds)")
display(tickers.sort_values("max_late", ascending=False).head(15)[cols].round(3))


In [ ]:
d = np.concatenate(late_sample) if late_sample else np.array([0.0])
fig, ax = plt.subplots(figsize=(9, 3.6))
ax.hist(np.log10(d + 1), bins=60, color="#c44e52")
ax.set_xlabel("log10(lateness in seconds + 1)"); ax.set_ylabel("late events (sampled)")
ax.set_title("Distribution of per-ticker lateness")
plt.tight_layout(); plt.show()

q = np.percentile(d, [50, 90, 99, 99.9])
pd.Series(q, index=["p50", "p90", "p99", "p99.9"], name="lateness_seconds").to_frame().round(1)


## 5. Per-ticker timelines (worst offenders)

A second pass reads only the worst tickers and plots each event time against its position in that ticker's stream, with the running watermark. Points below the red watermark line are the late events.

In [ ]:
worst = tickers.head(TOP_K).index.tolist()
seq = {i: [] for i in worst}

for b in read_batches(CSV_PATH):
    df = b.to_pandas()
    price = df[PRICE_COLS].ne("").any(axis=1).to_numpy()
    m = df["ID"].isin(worst).to_numpy() & price
    if not m.any():
        continue
    s = pd.to_timedelta(df.loc[m, "Time"], errors="coerce").dt.total_seconds().to_numpy()
    sub = pd.DataFrame({"ID": df.loc[m, "ID"].to_numpy(), "s": s})
    sub = sub[sub.s.notna() & (sub.s > 0)]     # drop missing and sentinel times
    for id_, grp in sub.groupby("ID", sort=False):
        seq[id_].extend(grp["s"].tolist())

fig, axes = plt.subplots(2, 3, figsize=(13, 6), sharex=False)
for ax, id_ in zip(axes.ravel(), worst):
    y = np.array(seq[id_]) / 3600.0
    x = np.arange(len(y))
    wm = np.maximum.accumulate(y)
    late = y < np.concatenate(([-np.inf], wm[:-1]))
    ax.plot(x, y, lw=0.6, color="#4c72b0", label="event time")
    ax.plot(x, wm, lw=0.9, color="#c44e52", label="ticker watermark")
    ax.scatter(x[late], y[late], s=4, color="#c44e52", zorder=3)
    ax.set_title(f"{id_}  ({int(tickers.loc[id_, 'n_late'])} late)", fontsize=9)
    ax.set_ylabel("hour of day")
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 6. Findings

In [ ]:
findings = pd.DataFrame([
    ("Scope", f"{len(tickers):,} tickers, {n_valid:,} price events analysed; {n_nonprice:,} non-price timed rows and {n_reference:,} reference rows ignored.",
     "Use this only as a per-ticker file-order diagnostic."),
    ("Coverage", f"{(tickers.n_late > 0).sum():,} tickers have at least one late event; "
                 f"{((tickers.n_late == 0) & (tickers.n_adj == 0)).sum():,} are fully time-ordered.",
     "Disorder is concentrated in a minority of tickers."),
    ("Volume", f"{int(tickers.n_late.sum()):,} late events ({100*tickers.n_late.sum()/tickers.n_events.sum():.2f}% of price events); "
               f"{int(tickers.n_adj.sum()):,} adjacent inversions.",
     "Compare with runtime trade and late-window counters before tuning Flink."),
    ("Magnitude", f"Worst lateness {tickers.max_late.max():,.0f} s; median sampled lateness {np.median(d):,.1f} s.",
     "Late events range from sub-second jitter to multi-hour stale snapshots."),
    ("Outliers", "One high system Time can raise the running maximum and flag many following rows.",
     "Inspect the timelines and adjacent inversions together."),
], columns=["aspect", "observation", "recommendation"])
findings


## 7. Runtime Replay Summary (Optional)

Set `DEBS_REPLAY_LOG` to the captured stdout from one completed Java `EventEmitter` replay of the same input (plain logs, without service prefixes). This cell reports the actual zone, filter, delivery result, first emitted event time (UTC), and loader accounting rather than reimplementing Java parsing. The first emitted event is not necessarily the minimum timestamp. These are producer-side counts, not Flink accepted-trade counts or EMA/window outcomes. A `DEBS_MAX_ROWS` prefix in the exploratory sections does not limit the captured run.

In [ ]:
import re

replay_log = os.getenv("DEBS_REPLAY_LOG")
if not replay_log:
    print("No actual replay summary: set DEBS_REPLAY_LOG to a completed emitter run's stdout.")
else:
    prefixes = {"configuration": "Replaying ", "producer result": "Done: ", "loader accounting": "Total: "}
    report = {}
    with open(replay_log, encoding="utf-8") as log:
        for line in log:
            for field, prefix in prefixes.items():
                if line.startswith(prefix) and (field != "configuration" or "(event zone " in line):
                    if field in report:
                        raise ValueError("DEBS_REPLAY_LOG must contain exactly one replay run")
                    report[field] = line.strip()
    if report.keys() != prefixes.keys():
        raise ValueError("DEBS_REPLAY_LOG is missing the configuration or completion summary")
    first_time = re.search(r"first event time (-?\d+)$", report["producer result"])
    if first_time and first_time.group(1) != "-1":
        report["first emitted event time (UTC)"] = pd.to_datetime(int(first_time.group(1)), unit="ms", utc=True)
    display(pd.Series(report, name="actual replay").to_frame())